## Universidad del Valle de Guatemala  
### Departamento de Ciencias de la Computacion  
### Inteligencia Artificial - Seccion 10
#### Nadissa Vela, 23764  
#### Cristian Tunchez, 231359

---

## Problemas de Satisfacción de Restricciones (CSP)

Desplegar 8 microservicios ($M_1$–$M_8$) en 3 servidores físicos ($S_1, S_2, S_3$) respetando:

1. **Restricción de Capacidad (Global):** Ningún servidor puede alojar más de 3 microservicios. Si se asignan 4, el peso de la asignación es 0.
2. **Restricciones de Anti-Afinidad (Binarias):** Las siguientes parejas no pueden estar en el mismo servidor:
   - $(M_1, M_2)$ — Base de datos primaria y su réplica
   - $(M_3, M_4)$
   - $(M_5, M_6)$
   - $(M_1, M_5)$

### Imports y Definición del Problema

In [9]:
import copy
from itertools import product

# Variables 
VARIABLES = ['M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8']

# Dominio de cada variable
DOMAIN = ['S1', 'S2', 'S3']

# Restricción de Capacidad
MAX_CAPACITY = 3

# Restricciones de Anti-Afinidad 
# Parejas que NO pueden estar en el mismo servidor
ANTI_AFFINITY = [
    ('M1', 'M2'),   # Base de datos primaria y su réplica
    ('M3', 'M4'),
    ('M5', 'M6'),
    ('M1', 'M5'),
]

print("Definición del Problema")
print(f"Variables   : {VARIABLES}")
print(f"Dominio     : {DOMAIN}")
print(f"Capacidad   : máx {MAX_CAPACITY} microservicios por servidor")
print(f"Anti-Afinidad: {ANTI_AFFINITY}")

Definición del Problema
Variables   : ['M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8']
Dominio     : ['S1', 'S2', 'S3']
Capacidad   : máx 3 microservicios por servidor
Anti-Afinidad: [('M1', 'M2'), ('M3', 'M4'), ('M5', 'M6'), ('M1', 'M5')]


### Funciones de Restricción (Constraints)

Estas funciones verifican si una asignación (completa o parcial) viola alguna restricción del problema.


In [ ]:
def check_anti_affinity(assignment):
    """
    Retorna True si NINGUNA pareja de anti-afinidad comparte servidor.
    Solo evalúa pares donde AMBAS variables están ya asignadas.
    """
    for (mi, mj) in ANTI_AFFINITY:
        if mi in assignment and mj in assignment:
            if assignment[mi] == assignment[mj]:
                return False
    return True


def check_capacity(assignment):
    """
    Retorna True si ningún servidor supera MAX_CAPACITY microservicios.
    """
    server_count = {}
    for server in assignment.values():
        server_count[server] = server_count.get(server, 0) + 1
    return all(count <= MAX_CAPACITY for count in server_count.values())


def compute_weight(assignment):
    """
    Peso de una asignación COMPLETA: 1 si es válida, 0 si viola alguna restricción.

    Definición formal:
        w(x) = 1  si  ∀(Mi, Mj) ∈ C_aa : x[Mi] ≠ x[Mj]
                     ∧ ∀ s ∈ S : |{ Mi : x[Mi] = s }| ≤ MAX_CAPACITY
        w(x) = 0  en caso contrario
    """
    if len(assignment) < len(VARIABLES):
        raise ValueError("compute_weight requiere una asignación completa.")
    if check_anti_affinity(assignment) and check_capacity(assignment):
        return 1
    return 0


# Prueba rápida
test_ok  = {'M1': 'S1', 'M2': 'S2', 'M3': 'S1', 'M4': 'S2',
            'M5': 'S2', 'M6': 'S1', 'M7': 'S3', 'M8': 'S3'}
test_bad = {'M1': 'S1', 'M2': 'S1', 'M3': 'S2', 'M4': 'S3',  # M1==M2 violación
            'M5': 'S2', 'M6': 'S3', 'M7': 'S1', 'M8': 'S2'}

print(f"test_ok  anti_affinity={check_anti_affinity(test_ok)},  capacity={check_capacity(test_ok)},  weight={compute_weight(test_ok)}")
print(f"test_badanti_affinity={check_anti_affinity(test_bad)}, capacity={check_capacity(test_bad)}, weight={compute_weight(test_bad)}")

### Función Heurística para Beam Search

Para el Beam Search necesitamos puntuar asignaciones parciales y elegir las mejores K. La heurística penaliza violaciones usando la fórmula:

$$w_{heuristic}(x) = 1 - \frac{\text{violaciones actuales}}{\text{total de restricciones}}$$


In [ ]:
def count_violations(assignment):
    """
    Cuenta el número total de violaciones en una asignación (parcial o completa).
      - Anti-afinidad: +1 por cada pareja prohibida que comparte servidor.
      - Capacidad: +1 por cada microservicio que excede MAX_CAPACITY en un servidor.
    """
    violations = 0

    # Anti-afinidad (solo evalúa pares donde ambas vars están asignadas)
    for (mi, mj) in ANTI_AFFINITY:
        if mi in assignment and mj in assignment:
            if assignment[mi] == assignment[mj]:
                violations += 1

    # Capacidad
    server_count = {}
    for server in assignment.values():
        server_count[server] = server_count.get(server, 0) + 1
    for count in server_count.values():
        if count > MAX_CAPACITY:
            violations += (count - MAX_CAPACITY)   # exceso sobre el límite

    return violations


# Normalizador: número total de restricciones del problema
TOTAL_CONSTRAINTS = len(ANTI_AFFINITY) + len(VARIABLES)


def heuristic_weight(assignment):
    """
    Heurística normalizada para Beam Search.

    Fórmula:
        w_h(x) = 1 - |V(x)| / |C_total|

    donde:
        V(x)    = número de restricciones violadas en x (parcial o completa)
        C_total = total de restricciones del problema (anti-afinidad + capacidad)

    Rango: [0, 1].  Mayor es mejor (menos violaciones).
    w_h(x) = 1  ↔  sin violaciones.
    w_h(x) < 1  ↔  al menos una restricción violada.
    """
    v = count_violations(assignment)
    return 1.0 - v / TOTAL_CONSTRAINTS


# Prueba
print(f"Sin violaciones  {heuristic_weight({'M1': 'S1', 'M2': 'S2'}):.4f}  (esperado ~1.0)")
print(f"Una violación    {heuristic_weight({'M1': 'S1', 'M2': 'S1'}):.4f}  (esperado <1.0)")
print(f"TOTAL_CONSTRAINTS = {TOTAL_CONSTRAINTS}")

---
## Task 2.1 
### Backtracking Search con Forward Checking (Lookahead)

El algoritmo de backtracking asigna valores a las variables una por una y, en cada paso, verifica que la asignación sea consistente con las restricciones. Para hacerlo más eficiente, utiliza forward checking, que al asignar un valor a una variable elimina de los dominios de las variables restantes aquellos valores que ya no serían válidos. Si esto provoca que algún dominio quede vacío, se descarta inmediatamente esa rama (poda) y se retrocede para probar otra opción. De esta forma, el algoritmo evita explorar soluciones imposibles desde etapas tempranas.

In [ ]:
def is_consistent(assignment, variable, value):
    """
    Comprueba si asignar variable=value es consistente con 'assignment'.
    Evalúa:
      1. Anti-afinidad frente a las variables ya asignadas.
      2. Capacidad: el servidor 'value' no puede superar MAX_CAPACITY.

    Formalmente:
        consistente(x, Xi, v) = True
            ↔  ∀(Xi, Xj) ∈ C_aa con Xj ∈ x : x[Xj] ≠ v
            ∧  |{ Xk ∈ x : x[Xk] = v }| < MAX_CAPACITY
    """
    # Anti-afinidad
    for (mi, mj) in ANTI_AFFINITY:
        if mi == variable and mj in assignment and assignment[mj] == value:
            return False
        if mj == variable and mi in assignment and assignment[mi] == value:
            return False

    # Capacidad
    current_count = sum(1 for v in assignment.values() if v == value)
    if current_count >= MAX_CAPACITY:
        return False

    return True


def forward_checking(assignment, variable, value, domains):
    """
    Aplica Forward Checking (Lookahead) después de asignar variable=value.

    Para cada variable sin asignar Xj, elimina de su dominio los valores que
    violarían una restricción con la asignación extendida:

        D_j ← D_j \\ { w ∈ D_j : (Xi=v, Xj=w) viola alguna restricción }

    Si D_j = ∅ para algún Xj → poda inmediata (rama sin solución posible).

    Retorna:
        new_domains  (dict)  si ningún dominio queda vacío.
        None                 si algún dominio queda vacío (poda).
    """
    new_domains = {k: list(v) for k, v in domains.items()}

    # Conteo actualizado de microservicios por servidor tras la nueva asignación
    server_count = {}
    for srv in list(assignment.values()) + [value]:
        server_count[srv] = server_count.get(srv, 0) + 1

    unassigned = [var for var in VARIABLES if var not in assignment and var != variable]

    for var in unassigned:
        to_remove = set()

        for val in new_domains[var]:
            # Eliminar val si viola anti-afinidad con la variable recién asignada
            for (mi, mj) in ANTI_AFFINITY:
                if (mi == variable and mj == var and val == value) or \
                   (mj == variable and mi == var and val == value):
                    to_remove.add(val)
                    break

            # Eliminar val si el servidor ya está lleno (restricción de capacidad)
            if val not in to_remove and server_count.get(val, 0) >= MAX_CAPACITY:
                to_remove.add(val)

        new_domains[var] = [val for val in new_domains[var] if val not in to_remove]

        if not new_domains[var]:
            return None  # Dominio vacío → poda esta rama

    return new_domains


def backtrack(assignment, domains, counter):
    """
    Búsqueda por Backtracking recursiva con Forward Checking.

    Algoritmo BT-FC(x, D):
        si |x| = |X|: retornar x si w(x) = 1, sino None
        Xi ← próxima variable no asignada (orden fijo izquierda→derecha)
        para cada v ∈ D[Xi]:
            si consistente(x, Xi, v):
                D' ← forward_checking(x, Xi, v, D)   # propagación de restricciones
                si D' ≠ None:                          # ningún dominio quedó vacío
                    x[Xi] ← v
                    resultado ← BT-FC(x, D')
                    si resultado ≠ None: retornar resultado
                    eliminar x[Xi]                     # backtrack
        retornar None

    Parámetros:
        assignment  dict   : asignación parcial actual.
        domains     dict   : dominios actuales de cada variable.
        counter     list   : [nodos_explorados] (mutable para contar desde fuera).

    Retorna la primera asignación completa válida encontrada, o None.
    """
    # Caso base: asignación completa
    if len(assignment) == len(VARIABLES):
        return assignment if compute_weight(assignment) == 1 else None

    # Seleccionar siguiente variable (orden fijo de izquierda a derecha)
    variable = next(v for v in VARIABLES if v not in assignment)

    for value in domains[variable]:
        counter[0] += 1  # contar nodo explorado

        if is_consistent(assignment, variable, value):
            # Aplicar Forward Checking: propagar reducción de dominios
            new_domains = forward_checking(assignment, variable, value, domains)

            if new_domains is not None:          # ningún dominio quedó vacío
                assignment[variable] = value
                result = backtrack(assignment, new_domains, counter)
                if result is not None:
                    return result
                del assignment[variable]         # backtrack

    return None


def backtracking_search():
    """
    Punto de entrada principal para Backtracking Search.
    Inicializa los dominios y lanza la búsqueda.
    """
    initial_domains = {var: list(DOMAIN) for var in VARIABLES}
    counter = [0]
    solution = backtrack({}, initial_domains, counter)
    return solution, counter[0]

In [13]:
def print_assignment(assignment, title="Asignación"):
    """Muestra una asignación de forma legible con distribución por servidor."""
    print(f"  {title}")
    if assignment is None:
        print("No se encontró solución.")
        return

    # Por microservicio
    for var in VARIABLES:
        print(f"  {var} → {assignment.get(var, '?')}")

    # Por servidor
    print()
    server_map = {s: [] for s in DOMAIN}
    for var, srv in assignment.items():
        server_map[srv].append(var)
    for srv in DOMAIN:
        services = server_map[srv]
        bar = '█' * len(services)
        print(f"  {srv} [{len(services)}/{MAX_CAPACITY}]  {bar}  {services}")

    violations = count_violations(assignment)
    weight     = compute_weight(assignment) if len(assignment) == len(VARIABLES) else "N/A (parcial)"
    print(f"\n  Violaciones : {violations}")
    print(f"  Weight      : {weight}")

# Ejecutar Backtracking Search
print("Ejecutando Backtracking Search con Forward Checking...\n")
bt_solution, bt_nodes = backtracking_search()

print_assignment(bt_solution, "Task 2.1 – Backtracking Search (primera solución válida)")
print(f"\n  Nodos explorados: {bt_nodes}")

Ejecutando Backtracking Search con Forward Checking...

  Task 2.1 – Backtracking Search (primera solución válida)
  M1 → S1
  M2 → S2
  M3 → S1
  M4 → S2
  M5 → S2
  M6 → S1
  M7 → S3
  M8 → S3

  S1 [3/3]  ███  ['M1', 'M3', 'M6']
  S2 [3/3]  ███  ['M2', 'M4', 'M5']
  S3 [2/3]  ██  ['M7', 'M8']

  Violaciones : 0
  Weight      : 1

  Nodos explorados: 8


---
## Task 2.2 
### Beam Search con tamaño de beam configurable K

Mantiene en cada paso solo los K mejores candidatos. 
Empieza con una asignación vacía y, al avanzar variable por variable, genera todas las posibles extensiones y se queda únicamente con las K que tienen mejor evaluación según una heurística (menos violaciones). 
Con K=1 se vuelve greedy (rápido pero menos preciso), y con K grande se acerca a una búsqueda exhaustiva (más precisa pero costosa).

In [ ]:
def beam_search(K=4):
    """
    Beam Search para el problema CSP de asignación de microservicios.

    Algoritmo:
        B_0 = { ∅ }                                              (candidato inicial vacío)
        Para i = 1..n  (una variable Xi por paso):
            E_i = { x ∪ {Xi: v} : x ∈ B_{i-1}, v ∈ D }        (expansión)
            B_i = top-K( E_i, clave = w_h(·) )                  (poda: conservar K mejores)
        Retornar argmax_{x ∈ B_n} w(x)   (preferir soluciones válidas)

    Casos límite:
        K = 1   → búsqueda greedy pura (rápida, puede perderse soluciones válidas).
        K = ∞   → BFS completo (garantiza encontrar solución si existe, costoso).

    Parámetros:
        K  int | None : tamaño del beam.
                        None (o float('inf')) = BFS sin poda.

    Retorna:
        best_assignment  dict  : la mejor asignación encontrada (puede tener violaciones
                                 si no existe ninguna válida con beam K).
        is_valid         bool  : True si la asignación retornada tiene weight = 1.
        candidates       list  : todos los candidatos finales (para inspección).
    """
    # Manejo de K infinito
    infinite = (K is None or K == float('inf'))

    # B_0: un único candidato – asignación vacía
    candidates = [{}]

    for variable in VARIABLES:
        # Expansión: E_i = { x ∪ {Xi: v} : x ∈ B_{i-1}, v ∈ D }
        extended = []
        for x in candidates:
            for value in DOMAIN:
                new_x = {**x, variable: value}
                extended.append(new_x)

        # Poda: ordenar por w_h descendente y conservar los K mejores
        extended.sort(key=heuristic_weight, reverse=True)

        if infinite:
            candidates = extended          # sin poda (BFS completo)
        else:
            candidates = extended[:K]      # conservar los K mejores según w_h

    # Seleccionar la mejor asignación final
    # Prioridad 1: asignaciones válidas (weight = 1)
    valid = [c for c in candidates if compute_weight(c) == 1]
    if valid:
        return valid[0], True, candidates

    # Si no hay ninguna válida, retornar la de menor número de violaciones
    best = min(candidates, key=count_violations)
    return best, False, candidates

In [15]:
# Ejecutar Beam Search con K = 4 (valor por defecto) 
print("Ejecutando Beam Search con K = 4...\n")
bs_solution, bs_valid, bs_candidates = beam_search(K=4)

print_assignment(bs_solution, "Task 2.2 – Beam Search  (K=4)")
print(f"\n  Solución válida encontrada: {bs_valid}")
print(f"  Candidatos finales en el beam: {len(bs_candidates)}")

Ejecutando Beam Search con K = 4...

  Task 2.2 – Beam Search  (K=4)
  M1 → S1
  M2 → S2
  M3 → S1
  M4 → S2
  M5 → S2
  M6 → S1
  M7 → S3
  M8 → S3

  S1 [3/3]  ███  ['M1', 'M3', 'M6']
  S2 [3/3]  ███  ['M2', 'M4', 'M5']
  S3 [2/3]  ██  ['M7', 'M8']

  Violaciones : 0
  Weight      : 1

  Solución válida encontrada: True
  Candidatos finales en el beam: 4


---
## Experimentos y Comparación

Probamos Beam Search con distintos valores de K y comparamos con Backtracking.

In [16]:
import time

# Backtracking Search 
t0 = time.perf_counter()
bt_sol, bt_nodes = backtracking_search()
bt_time = time.perf_counter() - t0
bt_violations = count_violations(bt_sol) if bt_sol else "N/A"
bt_valid = (compute_weight(bt_sol) == 1) if bt_sol else False

# Beam Search con varios valores de K 
K_values = [1, 2, 4, 8, None]   # None = sin poda (BFS completo)
beam_results = []

for K in K_values:
    t0 = time.perf_counter()
    sol, valid, cands = beam_search(K)
    elapsed = time.perf_counter() - t0
    violations = count_violations(sol)
    label = "∞" if K is None else str(K)
    beam_results.append({
        'K': label,
        'valid': valid,
        'violations': violations,
        'candidates': len(cands),
        'time_ms': elapsed * 1000,
        'solution': sol
    })

# Tabla de resultados 
print(f"\n{'─'*75}")
print(f"{'Algoritmo':<28} {'Válido':^8} {'Violaciones':^13} {'Nodos/K':^10} {'Tiempo (ms)':>12}")
print(f"{'─'*75}")
print(f"{'Backtracking + FC':<28} {'Sí' if bt_valid else 'No':^8} {str(bt_violations):^13} {str(bt_nodes):^10} {bt_time*1000:>10.3f}")
for r in beam_results:
    algo = f"Beam Search  K={r['K']}"
    print(f"{algo:<28} {'Sí' if r['valid'] else 'No':^8} {r['violations']:^13} {r['candidates']:^10} {r['time_ms']:>10.3f}")
print(f"{'─'*75}")

# Mostrar solución de cada Beam K
print("\n\n Detalle de asignaciones por K")
for r in beam_results:
    print_assignment(r['solution'], f"Beam Search  K={r['K']}")
    print(f"  Candidatos finales en beam : {r['candidates']}\n")


───────────────────────────────────────────────────────────────────────────
Algoritmo                     Válido   Violaciones   Nodos/K    Tiempo (ms)
───────────────────────────────────────────────────────────────────────────
Backtracking + FC               Sí          0           8           0.593
Beam Search  K=1                Sí          0           1           1.342
Beam Search  K=2                Sí          0           2           0.223
Beam Search  K=4                Sí          0           4           0.315
Beam Search  K=8                Sí          0           8           0.578
Beam Search  K=∞                Sí          0          6561        35.426
───────────────────────────────────────────────────────────────────────────


 Detalle de asignaciones por K
  Beam Search  K=1
  M1 → S1
  M2 → S2
  M3 → S1
  M4 → S2
  M5 → S2
  M6 → S1
  M7 → S3
  M8 → S3

  S1 [3/3]  ███  ['M1', 'M3', 'M6']
  S2 [3/3]  ███  ['M2', 'M4', 'M5']
  S3 [2/3]  ██  ['M7', 'M8']

  Violaciones : 

---
## Task 2.3
### Local Search Modos Condicionales Iterados (ICM)

El algoritmo ICM parte de una asignación completa aleatoria (que posiblemente viole restricciones) y la mejora iterativamente:

1. Inicializar $x$ con una asignación aleatoria completa.
2. Para cada variable $X_i$, calcular el peso de $x \cup \{X_i : v\}$ para cada valor $v \in D$.
3. Reasignar $X_i \leftarrow v^*$ donde $v^*$ maximiza dicho peso.
4. Repetir hasta que no haya mejora (convergencia) o se alcance el límite de iteraciones.

Propiedades:
- $\text{Weight}(x)$ nunca decrece entre iteraciones.
- Converge en un número finito de pasos.
- Puede quedar atrapado en **óptimos locales** — no garantiza la solución global.


In [ ]:
import random

def icm_search(max_iterations=1000, seed=None):
    """
    Búsqueda Local mediante Modos Condicionales Iterados (ICM).

    Algoritmo ICM:
        1. Inicializar x^(0) ~ Uniform(D^n)   (asignación completa aleatoria)
        2. Para cada variable Xi ∈ X (barrido completo):
               xi* = argmin_{v ∈ D} violations( x ∪ {Xi: v} )
                   ≡ argmax_{v ∈ D} w_h( x ∪ {Xi: v} )
               xi  ← xi*
        3. Repetir paso 2 hasta convergencia o alcanzar max_iterations.

    Propiedades formales:
        - Monotonía:   violations(x^(t+1)) ≤ violations(x^(t))
          La función objetivo nunca empeora entre iteraciones.
        - Convergencia: garantizada en tiempo finito (espacio de estados finito).
        - Optimalidad:  NO garantizada — puede converger a óptimos locales.

    Parámetros:
        max_iterations  int  : límite de iteraciones para evitar ciclos infinitos.
        seed            int  : semilla para reproducibilidad (None = aleatorio).

    Retorna:
        assignment   dict  : asignación final.
        is_valid     bool  : True si weight = 1.
        history      list  : violaciones por iteración (para visualizar convergencia).
        iterations   int   : número de iteraciones realizadas.
    """
    rng = random.Random(seed)

    # 1. Inicialización aleatoria completa: x^(0) ~ Uniform(D^n)
    assignment = {var: rng.choice(DOMAIN) for var in VARIABLES}

    history = [count_violations(assignment)]

    for iteration in range(max_iterations):
        improved = False

        # 2. Barrido completo: actualizar cada Xi según su modo condicional óptimo
        for variable in VARIABLES:
            current_value = assignment[variable]

            # xi* = argmin_{v ∈ D} violations(x ∪ {Xi: v})
            best_value = current_value
            best_violations = count_violations(assignment)

            for value in DOMAIN:
                if value == current_value:
                    continue
                # Evaluación tentativa: solo se cambia esta variable
                assignment[variable] = value
                v = count_violations(assignment)
                if v < best_violations:
                    best_violations = v
                    best_value = value

            # Aplicar el modo condicional óptimo
            assignment[variable] = best_value
            if best_value != current_value:
                improved = True

        history.append(count_violations(assignment))

        # 3. Convergencia: ninguna variable mejoró en este barrido completo
        if not improved:
            break

    is_valid = compute_weight(assignment) == 1
    return assignment, is_valid, history, iteration + 1

In [ ]:
# Ejecutar ICM con semilla fija para reproducibilidad
print("Ejecutando ICM (Modos Condicionales Iterados)...\n")
icm_sol, icm_valid, icm_history, icm_iters = icm_search(max_iterations=1000, seed=42)

print_assignment(icm_sol, "Task 2.3 – ICM Local Search")
print(f"\n  Iteraciones realizadas : {icm_iters}")
print(f"  Solución válida        : {icm_valid}")
print(f"  Historial de violaciones por iteración: {icm_history}")

# Comparar con múltiples semillas para mostrar sensibilidad al punto de inicio
print("\n\n Sensibilidad al punto inicial (10 ejecuciones con distintas semillas)")
print(f"{'─'*65}")
print(f"{'Semilla':^8} {'Válido':^8} {'Violaciones':^13} {'Iteraciones':^13} {'Inicio (viol.)':^14}")
print(f"{'─'*65}")

successes = 0
for s in range(10):
    sol, valid, hist, iters = icm_search(seed=s)
    v = count_violations(sol)
    if valid:
        successes += 1
    print(f"{s:^8} {'Sí' if valid else 'No':^8} {v:^13} {iters:^13} {hist[0]:^14}")

print(f"{'─'*65}")
print(f"  Tasa de éxito: {successes}/10 ejecuciones encontraron solución válida")
